# 01 — smolagents Basics

*Study Notebook 1 of 3 · about 30 minutes*

smolagents is Hugging Face's small library for building agents. Its core idea: **let the model write
Python to act, instead of forcing it to speak in rigid JSON.**

Roadmap:
1. **Step 2** — meet the two agent types (`CodeAgent`, `ToolCallingAgent`) and write your own tool
2. **Step 3** — one real task, solved both ways, side by side
3. **Step 4** — what JSON tool-calling looks like raw, and how much smolagents saves you
4. **Step 5** — when to use which
5. **Step 6** — your turn

You'll need a free Groq API key from [console.groq.com](https://console.groq.com) (no card required).

---
## Step 0 — Install

### Running locally on Windows? Set up a virtual environment first

*(Skip this if you're running in Google Colab — Colab already gives you an isolated environment.)*

Open **Command Prompt** or **PowerShell** in this repo's folder and run:

```powershell
py -3.12 -m venv .venv
.venv\Scripts\activate
python -m pip install --upgrade pip
pip install jupyter ipykernel
python -m ipykernel install --user --name foss7-agents --display-name "Python (foss7-agents)"
jupyter notebook
```

> PowerShell blocking `activate` with a script-execution error? Run this once, then retry:
> `Set-ExecutionPolicy -Scope Process -ExecutionPolicy Bypass`

Once Jupyter opens, reopen this notebook and switch the kernel: **Kernel → Change Kernel →
Python (foss7-agents)**. The `pip install` cell below then installs into `.venv` instead of your
system Python.

In [13]:
import sys
!{sys.executable} -m pip install -q "smolagents[toolkit]" litellm pandas

---
## Step 1 — Load your API key

Checks your environment, Colab Secrets, and a local `.env` file in order; only prompts if it finds
nothing. The key is never written into the notebook.

In [39]:
import os

def load_key(name: str):
    if os.environ.get(name):
        print(f"{name} loaded from environment"); return
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        print(f"{name} loaded from Colab Secrets"); return
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
        if os.environ.get(name):
            print(f"{name} loaded from .env file"); return
    except ImportError:
        pass

load_key("GROQ_API_KEY")
load_key("GEMINI_API_KEY")

# litellm.set_verbose = False


GROQ_API_KEY loaded from environment


---
## Step 2 — The two agent types

smolagents gives you two kinds of agent:
- **`CodeAgent`** — the model acts by writing Python
- **`ToolCallingAgent`** — the model acts by emitting JSON tool calls

We'll try both, then learn to write our own tool.

### 2.1 — Your first agent: `CodeAgent`

Ask it something a plain LLM gets wrong — a fact that changes over time — and give it a web-search tool.

- `CodeAgent` — acts by writing Python
- `WebSearchTool` — a built-in search tool
- `LiteLLMModel` — the bridge to Groq's free Llama 3.3

In [17]:
# !hf auth login

Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: FIQ7-8C7B

    Waiting for authorization......
Token is valid.
The token `oauth-ajmalbinnizam` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-ajmalbinnizam`
Note: This token will be refreshed automatically when it expires.


In [46]:
from smolagents import CodeAgent, WebSearchTool, LiteLLMModel

# Reused throughout the notebook
# model = LiteLLMModel(model_id="groq/llama-3.3-70b-versatile") # https://console.groq.com/keys
model = LiteLLMModel(model_id="gemini/gemini-2.5-flash") # https://aistudio.google.com/api-keys
model = LiteLLMModel(model_id="gemini/gemini-2.5-flash-lite") # https://aistudio.google.com/api-keys


# model = LiteLLMModel(model_id="groq/llama-3.1-8b-instant")

# smolagents is model-agnostic -- swap the line above for any of these:
# Hugging Face Inference (free, needs an HF token; may be blocked on some
# corporate networks -- that's why we default to Groq):
# from smolagents import InferenceClientModel
# model = InferenceClientModel(model_id="meta-llama/Llama-3.3-70B-Instruct")

#   Local, offline, no key (via Ollama):
#       model = LiteLLMModel(model_id="ollama_chat/llama3.2")
#   Anthropic / OpenAI / others via LiteLLM:
#

In [18]:
agent = CodeAgent(tools=[WebSearchTool()], model=model)
answer = agent.run("What is the latest stable version of Python, and when was it released?")
print("\nFINAL ANSWER:", answer)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest stable version of Python, and when was it released?                                          │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  latest_python_version = web_search(query="latest stable version of Python")                                      
  print("Latest stable version of Python:", latest_python_version)                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Latest stable version of Python: ## Search Results

[Download Python | Python.org](https://www.python.org/downloads/)
Looking for Python with a different OS? Python for Windows, Linux/Unix, macOS, Android, iOS, other Want to help 
test development versions  of  Python 3.15? Pre-releases, Docker images

[Python Release Python 3.14.6 | Python.org](https://www.python.org/downloads/latest/python3/)
A new command-line interface to inspect running Python processes using asynchronous tasks. The pdb module now 
supports remote attaching to a running Python process. For more details on the changes to Python 3.14, see What's 
new in Python 3.14. Build changes PEP 761: Python 3.14 and onwards no longer provides PGP signatures for release 
artifacts.

[Latest Python Version (2025) - What's New in Python 3.14?](https://www.liquidweb.com/blog/latest-python-version/)
Yes, Python 3.14 is the latest  stable release. We recommend testing this version in a staging or development 
environment before deploying in order to identify any refactoring required.

[Python Latest Version - Support Lifecycle & EOL](https://versionlog.com/python/)
 Python Lifecycle & End of Life (EOL) Policy Python follows a clear five-year support policy for every minor 
release. Once a version is fully released, it enters the bugfix phase where the core team accepts both bug fixes 
and security updates. New installer packages appear roughly every two months during this period to keep the 
ecosystem stable and reliable. After approximately two years, the ...

[Python 3.14: What Data Scientists and Developers Need to 
Know](https://www.anaconda.com/blog/python-3-14-what-data-scientists-developers-need-know)
 Python 3.14 brings Zstandard compression, multiple interpreters, free-threaded mode, and template strings. Learn 
what's new and how to prepare your workflows.

[Download and Install Python 3 Latest Version - 
GeeksforGeeks](https://www.geeksforgeeks.org/python/download-and-install-python-3-latest-version/)
Select the latest  Python 3 version , such as Python 3.13.1 (or whichever is the latest  stable  version 
available). Click on the version to download the installer (a .exe file for Windows).

[Python 3.14 Released and Other Python News for November 2025](https://realpython.com/python-news-november-2025/)
 Python 3.14 is officially out, Python 3.15 begins, and Python 3.9 reaches end of life. Plus, Django 6.0 first beta
released, new PEPs, and more Python news.

[The Latest Version of Python | phoenixNAP KB](https://phoenixnap.com/kb/latest-python-version)
The latest  stable  Python  version ensures better performance, improved compatibility with modern libraries, and 
ongoing security support. This guide will explain the current Python release, why updating matters, and how to 
check and upgrade your installed version .

[Python latest stable version? (2026) - 
solatatech.com](https://solatatech.com/article/python-latest-stable-version)
Which latest  Python  version is stable ? › The official latest  stable is 3.12, but I've experienced in the past 
that the library devs sometimes struggle to keep up with the published latest .

[How to Upgrade Python to the Latest 
Version](https://pythongeeks.net/python-tutorials/step-by-step-guide-to-update-python-version/)
Learn how to update Python in terminal, Windows, or macOS. Steps for upgrading Python  version , installing 
updates, and upgrading pip.

Out: None

[Step 1: Duration 1.56 seconds| Input tokens: 2,075 | Output tokens: 65]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The latest stable version of Python is 3.14, released on October 7, 2025.")                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The latest stable version of Python is 3.14, released on October 7, 2025.

[Step 3: Duration 0.76 seconds| Input tokens: 9,239 | Output tokens: 237]


FINAL ANSWER: The latest stable version of Python is 3.14, released on October 7, 2025.


In [19]:
# What it actually wrote to get that answer
from smolagents import ActionStep

system_prompt_step = agent.memory.system_prompt
print("The system prompt given to the agent was:")
print(system_prompt_step.system_prompt)

task_step = agent.memory.steps[0]
print("\n\nThe first task step was:")
print(task_step.task)

for step in agent.memory.steps:
    if isinstance(step, ActionStep):
        if step.error is not None:
            print(f"\nStep {step.step_number} got this error:\n{step.error}\n")
        else:
            print(f"\nStep {step.step_number} got these observations:\n{step.observations}\n")
        print(step.model_output)

The system prompt given to the agent was:
You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `fin

It wrote **Python** — a line like `web_search("latest Python version")` — not a JSON form.

### 2.2 — The other style: `ToolCallingAgent`

Same task, same tool — but switch the agent class. Now the model emits a **JSON** tool call instead of
writing code. Notice it's a one-line change.

In [20]:
from smolagents import ToolCallingAgent

tc_agent = ToolCallingAgent(tools=[WebSearchTool()], model=model)

try:
    answer = tc_agent.run("What is the latest stable version of Python?")
except Exception as e:
    print(f"Tool-calling hiccup ({type(e).__name__}) -- retrying once...")
    answer = tc_agent.run("What is the latest stable version of Python?")

print("\nFINAL ANSWER:", answer)

# litellm._turn_off_debug()

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the latest stable version of Python?                                                                    │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': 'latest stable version of Python'}                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: ## Search Results

|Download Python | Python.org](https://www.python.org/downloads/)
Looking for Python with a different OS? Python for Windows, Linux/Unix, macOS, Android, iOS, other Want to help 
test development versions  of  Python 3.15? Pre-releases, Docker images

|Python Releases for Windows](https://www.python.org/downloads/windows/)
The official home of the Python Programming Language

|Latest Python Version (2025) - What's New in Python 3.14?](https://www.liquidweb.com/blog/latest-python-version/)
Yes, Python 3.14 is the latest  stable release. We recommend testing this version in a staging or development 
environment before deploying in order to identify any refactoring required.

|Python Latest Version - Support Lifecycle & EOL](https://versionlog.com/python/)
 Python Lifecycle & End of Life (EOL) Policy Python follows a clear five-year support policy for every minor 
release. Once a version is fully released, it enters the bugfix phase where the core team accepts both bug fixes 
and security updates. New installer packages appear roughly every two months during this period to keep the 
ecosystem stable and reliable. After approximately two years, the ...

|Python 3.14: What Data Scientists and Developers Need to 
Know](https://www.anaconda.com/blog/python-3-14-what-data-scientists-developers-need-know)
 Python 3.14 brings Zstandard compression, multiple interpreters, free-threaded mode, and template strings. Learn 
what's new and how to prepare your workflows.

|Download and Install Python 3 Latest Version - 
GeeksforGeeks](https://www.geeksforgeeks.org/python/download-and-install-python-3-latest-version/)
Select the latest  Python 3 version , such as Python 3.13.1 (or whichever is the latest  stable  version 
available). Click on the version to download the installer (a .exe file for Windows).

|Python 3.14 Released and Other Python News for November 2025](https://realpython.com/python-news-november-2025/)
 Python 3.14 is officially out, Python 3.15 begins, and Python 3.9 reaches end of life. Plus, Django 6.0 first beta
released, new PEPs, and more Python news.

|Python latest stable version? (2026) - 
solatatech.com](https://solatatech.com/article/python-latest-stable-version)
Which latest  Python  version is stable ? › The official latest  stable is 3.12, but I've experienced in the past 
that the library devs sometimes struggle to keep up with the published latest .

|The Latest Version of Python | phoenixNAP KB](https://phoenixnap.com/kb/latest-python-version)
The latest  stable  Python  version ensures better performance, improved compatibility with modern libraries, and 
ongoing security support. This guide will explain the current Python release, why updating matters, and how to 
check and upgrade your installed version .

|Python 3.13 - What's New, Support Lifecycle & EOL - VersionLog](https://versionlog.com/python/3.13/)
What Is New in Python 3.13 Python 3.13 is a landmark release on two fronts: it ships the first official 
experimental free-threaded build (no GIL), and an experimental JIT compiler -- both opt-in. For everyday users, the
REPL gets a complete rewrite with multiline editing and color, the PEP 594 "dead batteries" are removed en masse, 
and import startup times improve across many stdlib modules.

[Step 1: Duration 2.06 seconds| Input tokens: 1,167 | Output tokens: 23]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Python 3.14'}                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Python 3.14

Final answer: Python 3.14

[Step 2: Duration 0.69 seconds| Input tokens: 3,145 | Output tokens: 46]


FINAL ANSWER: Python 3.14


**The difference in one line:**

| | `CodeAgent` | `ToolCallingAgent` |
|---|---|---|
| How it acts | Writes Python | Emits a JSON tool call |
| Good at | Logic: filtering, loops, combining results | One clean, single action per step |

Both are smolagents, both take the same tools, and switching between them is just the class name. Step 3
shows *why* the difference matters.

### 2.3 — Writing your own tool with `@tool`

A tool is just a Python function. The `@tool` decorator reads its type hints and docstring to build the
schema — no JSON, no config.

In [21]:
from smolagents import tool

@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text.

    Args:
        text: The text to count words in.
    """
    return len(text.split())

@tool
def char_count(text: str, include_spaces: bool = True) -> int:
    """Count characters in a piece of text.

    Args:
        text: The text.
        include_spaces: Whether to include spaces.
    """
    return len(text) if include_spaces else len(text.replace(" ", ""))

# They're ordinary functions too
print("Words:", word_count("code as actions"))

Words: 3


In [22]:
# An agent uses them from plain English
text_agent = CodeAgent(tools=[word_count, char_count], model=model)
print(text_agent.run(
    "For 'Kerala is God's own country', give the word count and the char count without spaces."
))

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ For 'Kerala is God's own country', give the word count and the char count without spaces.                       │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "Kerala is God's own country"                                                                             
  word_count_result = word_count(text)                                                                             
  char_count_result = char_count(text, include_spaces=False)                                                       
  print(f"Word count: {word_count_result}")                                                                        
  print(f"Char count without spaces: {char_count_result}")                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Word count: 5
Char count without spaces: 23

Out: None

[Step 1: Duration 1.64 seconds| Input tokens: 2,110 | Output tokens: 103]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Word count: 5, Char count without spaces: 23")                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Word count: 5, Char count without spaces: 23

[Step 2: Duration 0.98 seconds| Input tokens: 4,453 | Output tokens: 177]

Word count: 5, Char count without spaces: 23


In [23]:
# An agent uses them from plain English
text_agent = CodeAgent(tools=[word_count, char_count], model=model)
print(text_agent.run(
    "For 'Kerala is God's own country', give the word count and the char count without spaces."
))

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ For 'Kerala is God's own country', give the word count and the char count without spaces.                       │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.3-70B-Instruct ──────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "Kerala is God's own country"                                                                             
  word_count_result = word_count(text)                                                                             
  char_count_result = char_count(text, include_spaces=False)                                                       
  print("Word count:", word_count_result)                                                                          
  print("Char count without spaces:", char_count_result)                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Word count: 5
Char count without spaces: 23

Out: None

[Step 1: Duration 1.22 seconds| Input tokens: 2,110 | Output tokens: 106]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Word count: 5, Char count without spaces: 23")                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Word count: 5, Char count without spaces: 23

[Step 2: Duration 5.38 seconds| Input tokens: 4,451 | Output tokens: 157]

Word count: 5, Char count without spaces: 23


---
## Step 3 — The same call, two ways

### 3.1 — A marketing task

The marketing workflow at **Grand Hyatt Kochi** wants to send a coupon code to **female customers**.
We'll use 5 customers here; in the real world this would be 100+.

Same two tools, solved with `ToolCallingAgent` and then `CodeAgent`. Watch the step counts.

In [47]:
customers = [
    {"cust_id": "C1", "name": "subin", "gender": "M"},
    {"cust_id": "C2", "name": "ajmal", "gender": "M"},
    {"cust_id": "C3", "name": "sreejith", "gender": "M"},
    {"cust_id": "C4", "name": "vishnu", "gender": "M"},
    {"cust_id": "C5", "name": "aswathy achu", "gender": "F"},
]

def get_customers():
    """Return the list of customers."""
    return customers

def apply_coupon(cust_id, coupon_code):
    """Give a coupon to one customer."""
    return {"cust_id": cust_id, "coupon_code": coupon_code, "status": "applied"}

from smolagents import tool

@tool
def get_customers_tool() -> list:
    """Return the customer list. Each has cust_id, name, and gender."""
    return get_customers()

@tool
def apply_coupon_tool(cust_id: str, coupon_code: str) -> dict:
    """Give a coupon to one customer.

    Args:
        cust_id: Which customer.
        coupon_code: The coupon code.
    """
    return apply_coupon(cust_id, coupon_code)

hyatt_tools = [get_customers_tool, apply_coupon_tool]
job = "Apply coupon HYATT15 to every female customer. Report who received it."

### **Way 1 — `ToolCallingAgent` (JSON)**

In [49]:
from smolagents import ToolCallingAgent

json_agent = ToolCallingAgent(tools=hyatt_tools, model=model)
json_agent.run(job)
print(f"\nToolCallingAgent: {len(json_agent.memory.steps)} steps")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Apply coupon HYATT15 to every female customer. Report who received it.                                          │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_customers_tool' with arguments: {}                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: |{'cust_id': 'C1', 'name': 'subin', 'gender': 'M'}, {'cust_id': 'C2', 'name': 'ajmal', 'gender': 
'M'}, {'cust_id': 'C3', 'name': 'sreejith', 'gender': 'M'}, {'cust_id': 'C4', 'name': 'vishnu', 'gender': 'M'}, 
{'cust_id': 'C5', 'name': 'aswathy achu', 'gender': 'F'}]

[Step 1: Duration 1.33 seconds| Input tokens: 1,178 | Output tokens: 126]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Customer C5 (aswathy achu) received coupon HYATT15.'}  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Customer C5 (aswathy achu) received coupon HYATT15.

Final answer: Customer C5 (aswathy achu) received coupon HYATT15.

[Step 3: Duration 0.96 seconds| Input tokens: 5,851 | Output tokens: 459]


ToolCallingAgent: 4 steps


It fetches the customers, then calls `apply_coupon` once per female customer. The *"only female"*
rule can't ride inside a JSON argument, so the model checks each customer separately. Fine for 5;
costly for 100+.

### **Way 2 — `CodeAgent` (Python)**

In [51]:
from smolagents import CodeAgent

# pandas isn't in smolagents' default sandbox whitelist (only collections, datetime, itertools,
# math, queue, random, re, stat, statistics, time, unicodedata are auto-allowed) -- the model
# reaches for it naturally on tabular tasks like this one, so we authorize it explicitly.
code_agent = CodeAgent(tools=hyatt_tools, model=model, additional_authorized_imports=["pandas"])
code_agent.run(job)
print(f"\nCodeAgent: {len(code_agent.memory.steps)} steps")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Apply coupon HYATT15 to every female customer. Report who received it.                                          │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  customers = get_customers_tool()                                                                                 
  print(customers)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[{'cust_id': 'C1', 'name': 'subin', 'gender': 'M'}, {'cust_id': 'C2', 'name': 'ajmal', 'gender': 'M'}, {'cust_id': 
'C3', 'name': 'sreejith', 'gender': 'M'}, {'cust_id': 'C4', 'name': 'vishnu', 'gender': 'M'}, {'cust_id': 'C5', 
'name': 'aswathy achu', 'gender': 'F'}]

Out: None

[Step 1: Duration 1.64 seconds| Input tokens: 2,258 | Output tokens: 171]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  received_coupon = []                                                                                             
  for customer in customers:                                                                                       
      if customer['gender'] == 'F':                                                                                
          apply_coupon_tool(cust_id=customer['cust_id'], coupon_code='HYATT15')                                    
          received_coupon.append(customer['name'])                                                                 
  final_answer(f"The following female customers received the coupon 'HYATT15': {', '.join(received_coupon)}")      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The following female customers received the coupon 'HYATT15': aswathy achu

[Step 2: Duration 1.31 seconds| Input tokens: 4,717 | Output tokens: 308]


CodeAgent: 3 steps


In [ ]:
# The Python it wrote -- filter and loop in one action
for step in code_agent.memory.steps:
    if getattr(step, "model_output", None):
        print(step.model_output)
        break

### The takeaway

| | `ToolCallingAgent` (JSON) | `CodeAgent` (code) |
|---|---|---|
| Where the "female only" rule lives | Model's reasoning, per customer | One Python line |
| Applying the coupon | One JSON call per customer | One loop |
| Trips to the model | Grows with the list | One |
| Adding a rule (e.g. "and a repeat guest") | More steps | One more `and` |

The point isn't data size — JSON can carry plenty. It's that **conditions and loops don't fit in JSON
arguments**, so they're reasoned out call by call. `CodeAgent` puts that logic in code.

---
## Step 4 — What JSON tool-calling looks like raw

`ToolCallingAgent` did the JSON work for you: it wrote a schema per tool and ran the request/response
loop. Here's the same job **without** smolagents — using the OpenAI SDK against Groq (same protocol).
This is what smolagents saves you from typing.

In [52]:
import sys
!{sys.executable} -m pip install -q openai

In [53]:
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",  # Groq speaks the OpenAI protocol
)

# You write this schema by hand. smolagents generated it from @tool.
tools_schema = [
    {"type": "function", "function": {
        "name": "get_customers", "description": "Return the customer list.",
        "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {
        "name": "apply_coupon", "description": "Give a coupon to one customer.",
        "parameters": {"type": "object", "properties": {
            "cust_id": {"type": "string"}, "coupon_code": {"type": "string"}},
            "required": ["cust_id", "coupon_code"]}}},
]
FUNCTIONS = {"get_customers": get_customers, "apply_coupon": apply_coupon}

messages = [{"role": "user", "content": "Apply coupon HYATT15 to every female customer."}]
llm_calls = tool_calls = 0
for _ in range(12):
    llm_calls += 1
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile", messages=messages, tools=tools_schema)
    msg = resp.choices[0].message
    if not msg.tool_calls:
        print(f"\nDone: {llm_calls} LLM round trips, {tool_calls} tool calls.")
        break
    messages.append(msg)
    for tc in msg.tool_calls:
        tool_calls += 1
        args = json.loads(tc.function.arguments)
        out = FUNCTIONS[tc.function.name](**args)
        print(f"  round {llm_calls}: {tc.function.name}({args})")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(out)})

  round 1: apply_coupon({'coupon_code': 'HYATT15', 'cust_id': 'ID of female customer 1'})
  round 1: apply_coupon({'coupon_code': 'HYATT15', 'cust_id': 'ID of female customer 2'})


TypeError: __main__.get_customers() argument after ** must be a mapping, not NoneType

That's the raw version: a hand-written schema and a loop you maintain yourself. smolagents'
`ToolCallingAgent` gave you all of this from `@tool` plus one line — and `CodeAgent` skips the
per-action round trips entirely.

---
## Step 5 — When to reach for which

- **`CodeAgent`** — tasks with logic: filtering, loops, combining results, a little math. Most real work.
  It's the default.
- **`ToolCallingAgent`** — a single, well-defined action per step, or a model/setup that doesn't handle
  code well.

Quick head-to-head on the same task:

In [ ]:
task = "Count the words in each of: 'onam', 'vishu', 'thiruvathira'. Return all three."

ca = CodeAgent(tools=[word_count], model=model)
print("CodeAgent:", ca.run(task), "|", len(ca.memory.steps), "steps")

ta = ToolCallingAgent(tools=[word_count], model=model)
print("ToolCallingAgent:", ta.run(task), "|", len(ta.memory.steps), "steps")

Same answer; `CodeAgent` usually takes fewer steps by looping over the three words in one go.

---
## Step 6 — Your turn

A conditional task: give free shipping to orders above ₹2000. Run it with **both** agents and compare
the steps. Try adding a second rule (e.g. *"and placed in Kochi"*) and watch how each agent copes.

In [ ]:
orders = [
    {"order_id": "O1", "amount": 1500, "city": "Kochi"},
    {"order_id": "O2", "amount": 3200, "city": "Kochi"},
    {"order_id": "O3", "amount": 2500, "city": "Thrissur"},
    {"order_id": "O4", "amount": 900,  "city": "Kochi"},
]

@tool
def get_orders_tool() -> list:
    """Return the orders. Each has order_id, amount (INR), and city."""
    return orders

@tool
def grant_free_shipping_tool(order_id: str) -> dict:
    """Grant free shipping to one order.

    Args:
        order_id: Which order.
    """
    return {"order_id": order_id, "free_shipping": True}

shipping_tools = [get_orders_tool, grant_free_shipping_tool]
shipping_job = "Grant free shipping to every order above 2000 rupees. List the order ids."

# additional_authorized_imports=["pandas"]: same reason as the Hyatt CodeAgent above -- the model
# sometimes reaches for pandas on list-of-records tasks like this one.
ca = CodeAgent(tools=shipping_tools, model=model, additional_authorized_imports=["pandas"])
ca.run(shipping_job)
print(f"CodeAgent: {len(ca.memory.steps)} steps\n")

ta = ToolCallingAgent(tools=shipping_tools, model=model)
ta.run(shipping_job)
print(f"ToolCallingAgent: {len(ta.memory.steps)} steps")

---
## Why smolagents is good for prototyping

- **Small and readable.** The core is ~1,000 lines; little hidden magic.
- **Tools are just functions.** Type hints + docstring = a tool. No schema files (you saw the raw
  alternative in Step 4).
- **Model-agnostic.** Groq now, local Ollama or Claude later — one line changes.
- **Tool interop.** Pull tools from MCP servers, LangChain, or a Hugging Face Space.
- **Transparent.** `agent.memory.steps` shows the exact code written and each tool's result — easy to debug.
- **Safer code execution.** Generated code runs through a restricted executor, not raw `exec`. For
  production, use a real sandbox (Docker, E2B).

It optimises for getting an agent working fast and understanding it — ideal for prototyping, and it
pairs with LangGraph (next notebook) when you need structure.

---
## What to remember

| Idea | In one line |
|------|-------------|
| `CodeAgent` | Model writes Python — filters, loops, combines tools in one action |
| `ToolCallingAgent` | Model writes JSON — one clean action per step |
| JSON's real limit | Conditions and loops can't live in JSON args; they're reasoned out call by call |
| `@tool` | Type hints + docstring become the schema automatically |
| `LiteLLMModel` | One line to point smolagents at Groq, Anthropic, OpenAI, and more |

**Next:** `02_langgraph_basics.ipynb` — add structure and a human approval gate around agents like these.